### Window Functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
trans_df = spark.table("samples.bakehouse.sales_transactions")
display(trans_df)

In [0]:
trans_df.createOrReplaceTempView("transactions")

In [0]:
%sql
-- In Each FranchiseID, find out the second highest totalPrice product
with ranked_data as (
select 
*,
row_number() over (partition by franchiseID order by totalPrice desc) as rn
from transactions
)
select * from ranked_data where rn = 2;

In [0]:
%sql
-- In Each FranchiseID, find out the second highest totalPrice product
select 
*
from transactions
qualify row_number() over (partition by franchiseID order by totalPrice desc) = 2;

In [0]:
window = Window.partitionBy("franchiseID").orderBy(F.col("totalPrice").desc())
ranked_trans_df = (
    trans_df
    .withColumn(
        "rank",
        F.row_number().over(window)
    )
    .where(F.col("rank") == 2)
    .drop("rank")
)

ranked_trans_df.display()

In [0]:
bookings_df = spark.table("samples.wanderbricks.bookings")
bookings_df.display()

In [0]:
users_df = spark.table("samples.wanderbricks.users")
users_df.display()

In [0]:
# result_df = (
#     users_df.select("user_id", "name")
#     .join(
#         bookings_df,
#         on=["user_id"],
#         how="inner"

#     )
# )
# result_df.display()

In [0]:
filtered_bookings_df = (
    bookings_df
    .filter(F.col("status").isin("completed", "confirmed"))
    .select(
        "booking_id",
        "user_id",
        "property_id",
        "check_in",
        "check_out",
        "guests_count",
        "total_amount"
    )
)

window = Window.partitionBy("user_id")
ranked_df = (
    filtered_bookings_df
    .withColumn(
        "booking_count",
        F.count("booking_id").over(window)
    )
    .withColumn(
        "rn",
        F.row_number().over(window.partitionBy("user_id").orderBy(F.col("booking_count")))
    )
    .filter(F.col("rn") == 1)
    .withColumn("user_rank", F.dense_rank().over(Window.orderBy(F.col("booking_count").desc())))
    .filter(F.col("user_rank") <= 3)
)

ranked_df.display()

In [0]:
# Find out the top 3 users who have made hihest number of bookings, I want booking_id, user_id, user_name, email, booking_details... Consider only bookings which status is in completed or confirmed.

filtered_bookings_df = (
    bookings_df
    .filter(F.col("status").isin("completed", "confirmed"))
    .select(
        "booking_id",
        "user_id",
        "property_id",
        "check_in",
        "check_out",
        "guests_count",
        "total_amount"
    )
)

users_df = (
    users_df
    .select(
        "user_id", 
        "email", 
        F.col("name").alias("user_name")
    )
)

window = Window.partitionBy("user_id")
ranked_bookings_df = (
    filtered_bookings_df
    .withColumn(
        "booking_count",
        F.count("booking_id").over(window)
    )
    .withColumn(
        "rn",
        F.row_number().over(window.partitionBy("user_id").orderBy(F.col("booking_count")))
    )
    .filter(F.col("rn") == 1)
    .withColumn("user_rank", F.dense_rank().over(Window.orderBy(F.col("booking_count").desc())))
    .filter(F.col("user_rank") <= 3)
)


result_df = (
    ranked_bookings_df
    .join(
        users_df,
        on=["user_id"],
        how="inner"
    )
    .drop("rn", "user_rank")
)

result_df.display()
